# Sequential Counter with Ladder Encoding for SCAMO

Đây là một cách tiếp cận mới cho bài toán SCAMO (Staircase At Most One).

## Định nghĩa SCAMO
Định nghĩa: Cho $n$ biến boolean $\Omega = {x_1, x_2, \ldots, x_n}$ và một biến nguyên dương $w$ ($1 < w \leq n$), 1 SCAMO với $\Omega$ và $w$ là một công thức boolean có dạng:
$$\text{SCAMO}(\Omega, w) = \bigwedge_{i=0}^{n-w} \left( \bigvee_{j=i+1}^{i+w} x_j \leq 1 \right)$$

<center>
 <img src="./images/SCL/fig1.png" alt="SCAMO Example" width="600"/>
</center>

**Tính chất 1**: Ràng buộc $x_1 + x_2 + \ldots + x_n \leq 1$ sẽ được giữ nguyên khi và chỉ khi với $1 \leq i < n$: 
$$
\begin{aligned}
&(x_1+x_2+\ldots+x_i \leq 1) 
\land\ (x_{i+1}+x_{i+2}+\ldots+x_n \leq 1) \\
\land\ &\bigg( (x_1+x_2+\ldots+x_i \leq 0) \lor (x_{i+1}+x_{i+2}+\ldots+x_n \leq 0) \bigg)
\end{aligned}
$$

## Ý tưởng chính của SCL
Đối với SCAMO, ta chia thành các thành $M = \left\lceil \frac{n}{w} \right\rceil$ tập con. Mỗi tập con sẽ chứa một dãy biến boolean $\omega_i= {x_{i,1}, x_{i,2}, \ldots, x_{i,w}}$ với $1 \leq i \leq M$. Mỗi tập con $\omega_i$ với $1 < i <M$ sẽ lập ra 2 khối Sequential Counter (SC) với chiều ràng buộc ngược nhau. Cụ thể:
- Một khối SC sẽ có chiều ràng buộc từ trái sang phải là $B_i=(x_{i,1}+x_{i,2}+ \ldots + x_{i,w} \leq 1)$
- Một khối SC sẽ có chiều ràng buộc từ phải sang trái là $B_{i+1}=(x_{i,w}+x_{i,w-1}+ \ldots + x_{i,1} \leq 1)$.

Còn 2 tập con đầu tiên và cuối cùng sẽ chỉ có một khối SC với chiều ràng buộc ngược nhau. Cụ thể:
- Tập con đầu tiên sẽ chỉ có một khối SC với chiều ràng buộc từ phải sang trái là $B_1 = (x_{1,w}+x_{1,w-1}+ \ldots + x_{1,1} \leq 1)$.
- Tập con cuối cùng sẽ chỉ có một khối SC với chiều ràng buộc từ trái sang phải là $B_M = (x_{M,1}+x_{M,2}+ \ldots + x_{M,w} \leq 1)$.

Nếu tập con cuối cùng có số biến ít hơn $w$, ta không cần thêm các biến giả để đảm bảo số biến trong tập con cuối cùng bằng $w$ mà sử dụng trực tiếp số biến thực tế trong tập con cuối cùng.

Kết quả là ta có 2*(M-1) khối SC với chiều ràng buộc ngược nhau. 
<center>
 <img src="./images/SCL/fig2.png" alt="SCL Example" width="600"/>
</center>

### Các ràng buộc của SCL
Gọi $R_{i,j}$ là thanh ghi các bit với ý nghĩa là tổng của j biến đầu tiên. Gọi $x_{i,j}$ là biến boolean thứ $j$ trong khối $i$. Khi đó, ta có ý nghĩa với các giá trị sau của $R_{i,j}$:
- $R_{i,j} = \text{False}$ khi và chỉ khi $\sum_{j'=1}^{j} x_{i,j'} \leq 0$
- $R_{i,j} = \text{True}$ khi và chỉ khi $\sum_{j'=1}^{j} x_{i,j'} = 1$

Để đảm bảo tính đúng đắn của SCL, ta cần thiết lập các ràng buộc sau cho AMO và AMZ:
$$\bigwedge_{j=2}^{w} (x_{i,j} \rightarrow R_{i,j}) \tag{1}$$

$$\bigwedge_{j=2}^{w} (R_{i,j-1} \rightarrow R_{i,j}) \tag{2}$$

$$\bigwedge_{j=2}^{w} (\neg x_{i,j} \lor \neg R_{i-1,j} \rightarrow \neg R_{i,j}) \tag{3}$$

$$\bigwedge_{j=2}^{w} (x_{i,j} \rightarrow \neg R_{i,j-1}) \tag{4}$$

1. Ràng buộc (1): Nếu biến thứ $j$ trong khối $i$ được gán giá trị True, thì tổng của $j$ biến đầu tiên phải bằng 1.

2. Ràng buộc (2): Nếu tổng của $j-1$ biến đầu tiên bằng 1, thì tổng của $j$ biến đầu tiên cũng phải bằng 1.

3. Ràng buộc (3): Nếu biến thứ $j$ trong khối $i$ được gán giá trị False hoặc tổng của $j$ biến đầu tiên trong khối $i-1$ bằng 0, thì tổng của $j$ biến đầu tiên trong khối $i$ phải bằng 0.

4. Ràng buộc (4): Nếu biến thứ $j$ trong khối $i$ được gán giá trị True, thì tổng của $j-1$ biến đầu tiên phải bằng 0.

Trong đó, AMO (At Most One) được đảm bảo bởi cả 4 ràng buộc, còn AMZ (At Most Zero) được đảm bảo bởi 3 ràng buộc đầu tiên và ràng buộc thứ 4 được bỏ qua.

Tuy nhiên, đó chỉ là xây dựng các ràng buộc cho một khối SC. Để đảm bảo tính đúng đắn của toàn bộ SCL, ta cần thêm các ràng buộc liên kết giữa các khối SC để thoả mãn tính chất 1 của SCAMO. Cụ thể, với từng cặp khối SC liên tiếp đại diện cho ${x_i, x_{i+1}, \ldots, x_{w}}$ và ${x_{i+w+1}, x_{i+w+2}, \ldots, x_{i+w+w}}$, $0 \leq i \leq n-w$ ta cần thêm các ràng buộc sau:
$$
    \sum_{j=2}^{w} ((\sum_{k=i+j}^{i+w} x_{k} \leq 0) \lor (\sum_{k'=i+w+1}^{i+w+j-1} x_{k'} \leq 0))
$$

Từ công thức trên, ta có thể thay bằng các khối:
- $B_i$ được biểu diễn các ràng buộc với thanh ghi $R_{i}$ với độ dài là $l_i=w$ 
- $B_{i+1}$ được biểu diễn các ràng buộc với thanh ghi $R_{i+1}$ có độ dài là $l_{i+1} \leq w$ (vì khối cuối có thể có ít biến hơn $w$).

Với $1 \leq i < 2*(M-1)$ để rút gọn công thức thành:
$$
    \bigwedge_{t=1}^{min(w-1,l_{i+1})} (\neg R_{i,w-t} \lor \neg R_{i+1,t})
$$

Ví dụ, đối với $n=8$ và $w=4$, ta có công thức SCL với 2 tập con $\omega_1 = {x_1, x_2, x_3, x_4}$ và $\omega_2 = {x_5, x_6, x_7, x_8}$ như sau:
$$
\begin{gathered}
(x_1 + x_2 + x_3 + x_4 \le 1) \land \\
(x_2 + x_3 + x_4 \le 1) \land (x_5 \le 1) \land (x_2 + x_3 + x_4 \le 0 \lor x_5 \le 0) \land \\
(x_3 + x_4 \le 1) \land (x_5 + x_6 \le 1) \land (x_3 + x_4 \le 0 \lor x_5 + x_6 \le 0) \land \\
(x_4 \le 1) \land (x_5 + x_6 + x_7 \le 1) \land (x_4 \le 0 \lor x_5 + x_6 + x_7 \le 0) \land \\
(x_5 + x_6 + x_7 + x_8 \le 1)
\end{gathered}
$$
Thay bằng ký hiệu $R_{i,j}$ đối với các AMZ (AMO đã có các mệnh đề từ 4 ràng buộc nên không tiện để ký hiệu), ta có công thức SCL như sau:
$$
\begin{gathered}
(x_1 + x_2 + x_3 + x_4 \le 1) \land \\
(x_2 + x_3 + x_4 \le 1) \land (x_5 \le 1) \land (\neg R_{1,3} \lor \neg R_{2,1}) \land \\
(x_3 + x_4 \le 1) \land (x_5 + x_6 \le 1) \land (\neg R_{1,2} \lor \neg R_{2,2}) \land \\
(x_4 \le 1) \land (x_5 + x_6 + x_7 \le 1) \land (\neg R_{1,1} \lor \neg R_{2,3}) \land \\
(x_5 + x_6 + x_7 + x_8 \le 1)
\end{gathered}
$$

Nếu muốn giảm số biến phụ, ta có thể thay các $R_{i,1}$ bằng các biến $x_j$ có sẵn trong khối SC. Khi đó, công thức SCL sẽ được rút gọn như sau:

$$
\begin{gathered}
(x_1 + x_2 + x_3 + x_4 \le 1) \land \\
(x_2 + x_3 + x_4 \le 1) \land (x_5 \le 1) \land (\neg R_{1,3} \lor \neg x_5) \land \\
(x_3 + x_4 \le 1) \land (x_5 + x_6 \le 1) \land (\neg R_{1,2} \lor \neg R_{2,2}) \land \\
(x_4 \le 1) \land (x_5 + x_6 + x_7 \le 1) \land (\neg x_4 \lor \neg R_{2,3}) \land \\
(x_5 + x_6 + x_7 + x_8 \le 1)
\end{gathered}
$$

## Cài đặt SCL
Dưới đây là cài đặt SCL bằng Python:

In [13]:
from pysat.solvers import Glucose4
import math
class SCL:
    def __init__(self):
        self.total_variables = 0
        self.solver = Glucose4()
        self.variables = {}
        self.clauses = []
        
    
    def add_variable(self, name):
        self.variables[name] = self.total_variables + 1
        self.total_variables += 1
        return self.variables[name]
    
    def add_clause(self, clause):
        self.clauses.append(clause)
        self.solver.add_clause(clause)
    
    
    def split_into_subsets(self, variables,w):
        M = math.ceil(len(variables) / w)
        subsets = []
        for i in range(0,len(variables),w):
            if i + w > len(variables):
                subsets.append(variables[i:])
            else:
                subsets.append(variables[i:i+w])
        return subsets, M
    
    def create_block(self, subset, kind, direction, block_id):
        # direction: "LR" hoặc "RL"
        ordered = list(subset) if direction == "LR" else list(reversed(subset))

        regs = []
        # R_0 dùng luôn biến thật đầu tiên, không tạo biến phụ
        regs.append(ordered[0])

        for j in range(1, len(ordered)):
            regs.append(self.add_variable(f"R_{block_id}_{j}"))

        return {
            "id": block_id,
            "vars": ordered,
            "regs": regs,
            "kind": kind,          # "AMO" hoặc "AMZ"
            "direction": direction
        }


    def create_blocks(self, subsets):
        blocks = []

        for i, subset in enumerate(subsets):
            is_first = i == 0
            is_last = i == len(subsets) - 1

            if is_first:
                # block đầu cần suffix: x_w, x_{w-1}, ...
                blocks.append(
                    self.create_block(subset, "AMO", "RL", len(blocks))
                )

            elif is_last:
                # block cuối cần prefix: x_start, x_start+1, ...
                blocks.append(
                    self.create_block(subset, "AMO", "LR", len(blocks))
                )

            else:
                # block giữa cần prefix để nối với block trước
                blocks.append(
                    self.create_block(subset, "AMO", "LR", len(blocks))
                )

                # và cần suffix để nối với block sau
                blocks.append(
                    self.create_block(subset, "AMZ", "RL", len(blocks))
                )
                

        return blocks
    
    def encode_block(self, block):
        vars_ = block["vars"]
        regs = block["regs"]

        for j in range(1, len(vars_)):
            xj = vars_[j]
            rj = regs[j]
            rprev = regs[j - 1]

            # Formula 1: x_j -> R_j
            self.add_clause([-xj, rj])

            # Formula 2: R_{j-1} -> R_j
            self.add_clause([-rprev, rj])

            # Formula 3: ¬x_j ∧ ¬R_{j-1} -> ¬R_j
            self.add_clause([xj, rprev, -rj])

            # Formula 4 chỉ dùng cho AMO
            if block["kind"] == "AMO":
                self.add_clause([-xj, -rprev])

    def connect_blocks(self, blocks, w):
        # connect theo cặp: B0->B1, B2->B3, B4->B5...
        for i in range(0, len(blocks) - 1, 2):
            left = blocks[i]
            right = blocks[i + 1]

            left_regs = left["regs"]
            right_regs = right["regs"]

            limit = min(w - 1, len(right_regs))

            for t in range(1, limit + 1):
                self.add_clause([
                    -left_regs[w - t - 1],
                    -right_regs[t - 1]
                ])

    def scamo(self, variables, w):
        if not (1 < w <= len(variables)):
            raise ValueError("SCAMO requires 1 < w <= len(variables)")
        subsets, _ = self.split_into_subsets(variables, w)

        blocks = self.create_blocks(subsets)

        for block in blocks:
            self.encode_block(block)

        self.connect_blocks(blocks, w)
        
    

In [14]:
from itertools import product

def check_scamo_semantics(bits, w):
    n = len(bits)

    for i in range(n - w + 1):
        if sum(bits[i:i+w]) > 1:
            return False

    return True

from pysat.solvers import Glucose4

def verify_encoder(n, w):
    scl = SCL()

    vars_ = [scl.add_variable(f"x{i}") for i in range(n)]

    scl.scamo(vars_, w)

    clauses = scl.clauses

    for assignment in product([False, True], repeat=n):

        # semantic thật
        expected = check_scamo_semantics(assignment, w)

        # build SAT solver mới
        solver = Glucose4()

        for c in clauses:
            solver.add_clause(c)

        # fix assignment
        for var, val in zip(vars_, assignment):
            solver.add_clause([var if val else -var])

        sat = solver.solve()

        if sat != expected:
            print("Mismatch!")
            print("assignment =", assignment)
            print("expected =", expected)
            print("sat =", sat)
            return False

    return True

print(verify_encoder(5, 2))
print(verify_encoder(6, 3))
print(verify_encoder(7, 4))

True
True
True
